In [47]:
# 创建 chat model
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage
from langgraph.checkpoint.memory import InMemorySaver  # 短期记忆 demo 级别记录短期记忆
from langgraph.store.memory import InMemoryStore  # 长期记忆 demo 级别记录长期记忆


load_dotenv()

aliyun_base_url = os.getenv("ALIYUN_BASE_URL")
aliyun_api_key = os.getenv("ALIYUN_API_KEY")

model = init_chat_model(
    model="deepseek-v4-flash-0731",
    model_provider="openai",
    base_url=aliyun_base_url,
    api_key=aliyun_api_key,
)

qwen38_model = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    base_url=aliyun_base_url,
    api_key=aliyun_api_key,
)


In [19]:
agent = create_agent(
    model=model,
    system_prompt="you are a good friend",
    checkpointer=InMemorySaver(),  # 为 checkpointer 指定一个内存存储器，可以存储当前会话的短期记忆
    store=InMemoryStore(),  # store 指定一个内存储存来记录长期记忆
)

thread_config = {"configurable": {"thread_id": "thread_1"}} # thread_config 用来管理当前的会话信息配置，这对短期记忆来说很重要

stream = agent.stream_events(
    {
        "messages": [HumanMessage("杭州天气怎么样?")],
    },
    version="v3",
    config = thread_config,
)


for message in stream.messages:
    for delta in message.text:
        print(delta, end="", flush=True)

final_state = stream.output

print("\n\nfinal_state: ", final_state)

I'd love to help you with the weather in Hangzhou!

However, I don't have real-time access to current weather data. To get the most accurate and up-to-date forecast, I'd recommend checking a reliable weather service like the China Meteorological Administration, or apps like AccuWeather, Weather.com, or the local weather app on your phone.

If you have a specific date in mind or want to know about typical weather patterns for a certain season in Hangzhou (like "what's fall weather usually like?"), I'd be happy to give you general information! Just let me know. 😊

final_state:  {'messages': [HumanMessage(content='杭州天气怎么样?', additional_kwargs={}, response_metadata={}, id='6d33475a-eb17-48cb-8266-95ccd53a5ce4'), AIMessage(content=[{'type': 'text', 'text': 'I\'d love to help you with the weather in Hangzhou!\n\nHowever, I don\'t have real-time access to current weather data. To get the most accurate and up-to-date forecast, I\'d recommend checking a reliable weather service like the China M

In [20]:
agent = create_agent(
    model=model,
    system_prompt="you are a good friend",
)

thread_config = {"configurable": {"thread_id": "thread_1"}} # thread_config 用来管理当前的会话信息配置，这对短期记忆来说很重要

stream = agent.stream_events(
    {
        "messages": [HumanMessage("杭州天气怎么样?")],
    },
    version="v3",
    config = thread_config,
)

for message in stream.messages:
    print(f"[{message.node}] ", end="")
    # message.text 消息的文本增量与最终文本。
    for delta in message.text:
        print(delta, end="", flush=True)
    # stream.messages生成 ChatModelStream 对象
    # message.output 模型调用完成后的最终消息对象。
    full_message = message.output
    usage = full_message.usage_metadata
    if usage:
        print("\n\nusage:: ", usage)

[model] 哈哈，你这个问题可把我难住了！因为我是一个AI模型，没有实时联网功能，所以**没法直接告诉你杭州此时此刻的天气**哦。

不过，我可以给你一些实用的建议：

1.  **最准确的方法**：打开你手机上的**天气App**，或者搜索“杭州天气”，就能看到实时温度、湿度和未来几天的预报。
2.  **如果想知道穿衣建议**：你可以告诉我你**计划什么时候去杭州**（比如是下个月还是明天），我可以根据杭州那个季节的大致气候，给你一些穿衣和出行的参考建议。

需要我帮你查一下杭州某个季节的天气特点吗？随时问我！

usage::  {'input_tokens': 92, 'output_tokens': 265, 'total_tokens': 357, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 128}}


In [74]:
from langchain.tools import tool


@tool
def get_weather(city: str) -> str:
    """Get weather for a city."""
    return f"It's always sunny in {city}!"


agent = create_agent(
    model=qwen38_model,
    tools=[get_weather],
    system_prompt="you are a good friend",
)

thread_config = {
    "configurable": {"thread_id": "thread_2"}
}  # thread_config 用来管理当前的会话信息配置，这对短期记忆来说很重要

stream = agent.stream_events(
    {
        "messages": [HumanMessage("杭州天气怎么样?")],
    },
    version="v3",
    config=thread_config,
)

for message in stream.messages:
    for chunk in message.tool_calls:
        print(f"tool call chunk: {chunk}")

    finalized = message.tool_calls.get()
    if finalized:
        print(f"finalized tool calls: {finalized}")

for call in stream.tool_calls:
    print(f"{call.tool_name}({call.input})")
    for delta in call.output_deltas:
        print(delta, end="", flush=True)
    print(call.output, call.error)

tool call chunk: {'type': 'tool_call_chunk', 'id': 'call_ceedd6abe4d7438ca5cad17a', 'name': 'get_weather', 'args': '', 'index': 0}
tool call chunk: {'type': 'tool_call_chunk', 'id': 'call_ceedd6abe4d7438ca5cad17a', 'name': 'get_weather', 'args': '{"city": "杭州', 'index': 0}
tool call chunk: {'type': 'tool_call_chunk', 'id': 'call_ceedd6abe4d7438ca5cad17a', 'name': 'get_weather', 'args': '{"city": "杭州"', 'index': 0}
tool call chunk: {'type': 'tool_call_chunk', 'id': 'call_ceedd6abe4d7438ca5cad17a', 'name': 'get_weather', 'args': '{"city": "杭州"}', 'index': 0}
finalized tool calls: [{'type': 'tool_call', 'id': 'call_ceedd6abe4d7438ca5cad17a', 'name': 'get_weather', 'args': {'city': '杭州'}}]


当主 Agent 通过工具去调用另一个“具名（Named）”子 Agent 时，框架会自动将子 Agent 的内部运行轨迹、消息和状态隔离并打包好，让你能够极其方便地在外部（如日志或前端 UI 中）精准追踪是谁在干活、干了什么。

核心亮点与作用拆解
命名隔离与身份识别 (name=)

当你为子 Agent 设置了 name="weather_agent" 后，框架在底层事件流中会给它打上专属的标签。这解决了多 Agent 系统中最头疼的“日志混淆”问题，让你一眼看出哪条消息是哪个 Agent 产出的。

专属的监控投影流 (stream.subagents)

你不需要在海量的、杂乱的底层事件（Event Stream）里费力过滤。系统提供了一个独立的 stream.subagents 视图，专门用来收集所有被调用的命名子 Agent。

丰富的结构化上下文 (.messages, .cause 等)

每个子 Agent 的句柄（Handle）直接暴露了丰富的数据：

.messages / .output：子 Agent 内部的对话历史和最终输出。

自动过滤杂讯

框架只抓取通过 create_agent 创建的、带有名字的子 Agent 运行实例，自动帮你把底层普通的、繁琐的子图（Subgraphs）过滤掉，只留下干净、高层的 Agent 级交互。

In [45]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.messages import HumanMessage

@tool("get_weather", description="Get weather for a given city.")
def get_weather_of_sub_agent(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

weather_agent = create_agent(
    model=qwen38_model,
    tools=[get_weather_of_sub_agent],
    name="weather_agent",
)

@tool
def call_weather(city: str) -> str:
    """根据给定的城市查询天气"""
    result = weather_agent.invoke(
        {"messages": [{"role": "user", "content": f"{city}天气如何？"}]}
    )
    content = result["messages"][-1].content
    print(f"\n[Sub-Agent 内部捕获]: {content}") # 直接在工具执行处就能实时拿到
    return content

supervisor = create_agent(
    model=qwen38_model,
    tools=[call_weather],
    name="supervisor",
)

stream = supervisor.stream_events(
    {"messages": [HumanMessage("今天北京的天气如何？")]},
    version="v3",
)

for subagent in stream.subagents:
    print(f"{subagent.name}: ", end="")
    print("\n\nsubagent.messages messages:")
    for message in subagent.messages:
        for token in message.text:
            print(token, end="", flush=True)
    print()

weather_agent: 

subagent.messages messages:
北京现在是晴天 ☀️

如果你需要更详细的天气信息（比如气温、风力、湿度或未来几天的预报），也可以告诉我，我来帮你查询。
[Sub-Agent 内部捕获]: [{'type': 'text', 'text': '北京现在是晴天 ☀️\n\n如果你需要更详细的天气信息（比如气温、风力、湿度或未来几天的预报），也可以告诉我，我来帮你查询。', 'index': 0}]



LangChain 实现了一套流式传输系统，用于展示实时更新信息。
流式传输对于提升基于大语言模型构建的应用的响应能力至关重要。它可以在完整回复生成完成前，逐步展示输出内容，大幅优化用户体验，在应对大语言模型存在延迟的场景下效果尤为突出。

LangChain 的流式系统支持将智能体运行过程中的实时反馈输出到你的应用中。
LangChain 流式能力可实现的功能：
 流式输出智能体进度——在智能体每一步执行后获取状态更新。
 流式输出大模型令牌——在语言模型生成令牌的同时进行流式推送。
 流式输出思考/推理令牌——实时展示模型正在生成的推理内容。
 流式输出自定义更新信息——发送用户自定义信号（例如，"Fetched 10/100 records"）。
 多模式流式输出——可选择 updates（智能体进度）、messages（大模型令牌+元数据）或者 custom（任意用户数据）模式。


如需流式输出智能体执行进度，请使用带 stream 或 astream 方法并配置 stream_mode="updates"。该配置会在智能体每完成一步后推送事件。
例如，若你的智能体调用一次工具，将会收到如下更新：
大模型节点：AIMessage，携带工具调用请求
工具节点：ToolMessage，携带执行结果
大模型节点：最终AI回复
通过 config 传入 thread_id，对话会保存检查点，后续轮次可以接续同一段对话历史。thread_id 和 stream_mode 相互独立；你也可以同步传入 context，用于传递单次运行数据，工具可从 runtime.context 读取该数据。

In [66]:
import asyncio
from langchain_core.utils.uuid import uuid7


@tool
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"


agent = create_agent(
    model=qwen38_model, tools=[get_weather], checkpointer=InMemorySaver()
)

config = {"configurable": {"thread_id": str(uuid7())}}

stream = await agent.astream_events(
    {
        "messages": [HumanMessage("杭州天气怎么样?")],
    },
    version="v3",
    config=config,
)


async def consume_messages():
    async for message in stream.messages:
        txt = await message.text
        if txt:
            print("message.text: ",txt )


async def consume_tool_calls():
    async for call in stream.tool_calls:
        print("\n\nconsume_tool_calls:", call.tool_name, call.input)


results = await asyncio.gather(consume_messages(), consume_tool_calls())



consume_tool_calls: get_weather {'city': '杭州'}
message.text:  杭州现在是晴天 ☀️

不过我获取到的信息比较简略，没有具体的温度、湿度和风力数据。建议你查看手机天气 App 或当地气象网站，获取更详细的实时天气和未来几天的预报。


In [69]:
from langchain_core.utils.uuid import uuid7
from langgraph.checkpoint.memory import InMemorySaver


@tool
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

agent = create_agent(
    model=qwen38_model, tools=[get_weather], checkpointer=InMemorySaver()
)

config = {"configurable": {"thread_id": str(uuid7())}}

stream = agent.stream_events(
    {"messages": [{"role": "user", "content": "杭州天气怎样"}]},
    config=config,
    version="v3",
)
# stream.interleave("messages", "tool_calls") 的核心作用是：将多个不同的数据流（比如大模型的对话消息流 messages 和工具调用流 tool_calls）按照时间顺序“交织（Interleave）”并合并成一个单一的迭代器，
# 让你在同步循环中能够按顺序、无阻塞地同时处理它们
for kind, item in stream.interleave("messages", "tool_calls"):
    if kind == "messages":
        for token in item.text:
            print(token, end="", flush=True)
    elif kind == "tool_calls":
        print(f"\nTool call: {item.tool_name}({item.input})")
        for delta in item.output_deltas:
            print(delta, end="", flush=True)
        print(f"\nTool result: {item.output}")


Tool call: get_weather({'city': '杭州'})

Tool result: It's always sunny in 杭州!
杭州目前是晴天，阳光充足！☀️ 适合外出活动，不过晴天也要注意防晒哦。

（提示：这个天气工具提供的信息比较简单，如果你需要气温、湿度、未来几天的预报等更详细的天气信息，建议查看专业的气象应用或网站。）

三种核心 Stream Mode 的作用详解
"updates"（状态更新模式）

作用：在每个 Agent 执行步骤（step）结束后，流式返回状态的更新情况。如果在一个步骤中有多个节点被连续运行，这些更新会分别独立流出。

适用场景：适合用来追踪工作流的执行进度、观察每一步状态（State）发生了什么变化。

"messages"（消息/Token 流模式）

作用：当图中的任何节点调用了 LLM（大模型）时，它会实时流式返回 (token, metadata) 元组。

适用场景：这是实现对话界面“打字机效果”（流式吐字）最常用的模式。

"custom"（自定义数据模式）

作用：允许你在图的节点内部，通过特定的流写入器（stream writer）主动推送自定义的数据。

适用场景：当你需要传递一些大模型 Token 或状态更新之外的内容（例如：业务进度百分比、自定义日志、中间计算制品等）时使用。

In [77]:
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"


agent = create_agent(
    model=qwen38_model, tools=[get_weather], checkpointer=InMemorySaver()
)
config = {"configurable": {"thread_id": str(uuid7())}}
stream = agent.stream_events(
    {"messages": [{"role": "user", "content": "杭州天气怎样?"}]},
    config=config,
    version="v3",
)

#  stream.interleave 将 多个事件流合并在一起
for kind, item in stream.interleave("messages", "tool_calls"):
    if kind == "messages":
        for token in item.text:
            print(token, end="", flush=True)
    elif kind == "tool_calls":
        print(f"\nTool call: {item.tool_name}({item.input})")
        for delta in item.output_deltas:
            print(delta, end="", flush=True)
        print(f"\nTool result: {item.output}")

final_state = stream.output


Tool call: get_weather({'city': '杭州'})

Tool result: It's always sunny in 杭州!
杭州现在是晴天，阳光明媚！☀️

不过我获取到的信息比较简略，如果你需要了解具体的温度、湿度或未来几天的天气预报，建议查看专业的天气 App 或网站获取更详细的数据。

In [ ]:
# 流式输出打字效果 使用 agent.stream + stream_mode="messages" 打字效果
@tool
def get_weather(city: str) -> str:
    """Get weather for a given city."""

    return f"It's always sunny in {city}!"

agent = create_agent(
    model=qwen38_model,
    tools=[get_weather],
)

# stream 方法：粗粒度（侧重状态与节点）
# 以“图的节点（Node）执行完毕”或“状态更新（Updates）”为单位，或者是基于传统 StateGraph 的整体状态输出
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="messages",
    version="v2",
):
    if chunk.get("type") == "messages":
        token, metadata = chunk["data"]
        
        # 针对 token 内容进行判断并直接打印文本
        if hasattr(token, "content") and token.content:
            print(token.content, end="", flush=True)
        elif hasattr(token, "content_blocks") and token.content_blocks:
            # 如果 content_blocks 存在且有内容，遍历并打印文本块
            for block in token.content_blocks:
                if isinstance(block, dict) and block.get("type") == "text":
                    print(block.get("text", ""), end="", flush=True)
                elif isinstance(block, str):
                    print(block, end="", flush=True)

It's always sunny in San Francisco!The weather tool returned: **"It's always sunny in San Francisco!"**

That looks like a canned/joke response rather than real weather data, so I wouldn't rely on it for planning. If you want accurate current conditions, check a live source like weather.gov or a weather app. Happy to try the tool for a different city if you'd like.

In [57]:
# 要在工具执行时流式推送更新，你可以使用 get_stream_writer。

from langgraph.config import get_stream_writer


@tool
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    writer = get_stream_writer()
    # stream any arbitrary data
    writer(f" step1 判断是否是正确的城市: {city}")
    writer(f" step2 获取城市的天气预报数据: {city}")
    return f"It's always sunny in {city}!"


agent = create_agent(
    model=qwen38_model,
    tools=[get_weather],
)


for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="custom", # 指定使用自定义流
    version="v2",
):
    print("type: ", chunk["type"])
    if chunk["type"] == "custom":
        print(chunk["data"])

type:  custom
 step1 判断是否是正确的城市: San Francisco
type:  custom
 step2 获取城市的天气预报数据: San Francisco


In [ ]:
from langgraph.config import get_stream_writer

# 自定义流式输出 需要使用 writer = get_stream_writer() 后， writer(description) 可以输出自定义数据 

@tool
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    writer = get_stream_writer()
    # stream any arbitrary data
    writer(f" step1 判断是否是正确的城市: {city}")
    writer(f" step2 获取城市的天气预报数据: {city}")
    return f"It's always sunny in {city}!"


agent = create_agent(
    model=qwen38_model,
    tools=[get_weather],
)


for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["custom", "updates"],
    version="v2",
):
    print("type: ", chunk["type"], " chunk: ", chunk)
    if chunk["type"] == "custom":
        print(chunk["data"])

type:  updates  chunk:  {'type': 'updates', 'ns': (), 'data': {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 322, 'total_tokens': 376, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 26, 'rejected_prediction_tokens': None, 'text_tokens': 54}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': 322}}, 'model_provider': 'openai', 'model_name': 'qwen3.8-max', 'system_fingerprint': None, 'id': 'chatcmpl-93949719-aa50-97a4-92ab-f4a12fc320c0', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a176fc-87b2-7350-97d4-acf956550043-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'SF'}, 'id': 'call_82e7aa6c4143495da48d1164', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 322, 'output_tokens': 54